In [1]:
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42

# =========================
# 1. 경로 설정
# =========================

DATA_DIR = Path("data")
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 입력 파일
fnb_success_path = DATA_DIR / "fnb_shorts_success_data.csv"
fnb_fail_path    = DATA_DIR / "fnb_shorts_fail_data.csv"
it_success_path  = DATA_DIR / "it_shorts_success_data.csv"
it_fail_path     = DATA_DIR / "it_shorts_fail_data.csv"

# =========================
# 2. 데이터 로드
# =========================

df_fnb_success = pd.read_csv(fnb_success_path, encoding="utf-8")
df_fnb_fail    = pd.read_csv(fnb_fail_path, encoding="utf-8")
df_it_success  = pd.read_csv(it_success_path, encoding="utf-8")
df_it_fail     = pd.read_csv(it_fail_path, encoding="utf-8")


# =========================
# 3. 필수 컬럼 확인
# =========================

required_cols = ["video_id", "final_url", "채널명"]

def check_required_cols(df, name):
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{name}에 필수 컬럼이 없습니다: {missing}")

check_required_cols(df_fnb_success, "df_fnb_success")
check_required_cols(df_fnb_fail, "df_fnb_fail")
check_required_cols(df_it_success, "df_it_success")
check_required_cols(df_it_fail, "df_it_fail")


# =========================
# 4. 도메인/성공여부 컬럼 추가
# =========================

df_fnb_success = df_fnb_success.copy()
df_fnb_fail    = df_fnb_fail.copy()
df_it_success  = df_it_success.copy()
df_it_fail     = df_it_fail.copy()

df_fnb_success["domain"] = "FnB"
df_fnb_fail["domain"]    = "FnB"
df_it_success["domain"]  = "IT"
df_it_fail["domain"]     = "IT"

df_fnb_success["success_label"] = "success"
df_fnb_fail["success_label"]    = "fail"
df_it_success["success_label"]  = "success"
df_it_fail["success_label"]     = "fail"


# =========================
# 5. 채널별 cap 적용 함수
# =========================

def apply_channel_cap(df, cap=10, random_state=42):
    """
    각 채널별로 최대 cap개까지만 남기는 함수
    """
    sampled_list = []

    for channel, group in df.groupby("채널명"):
        n = min(len(group), cap)
        sampled = group.sample(n=n, random_state=random_state)
        sampled_list.append(sampled)

    if not sampled_list:
        return pd.DataFrame(columns=df.columns)

    return pd.concat(sampled_list, ignore_index=True)


# =========================
# 6. 비례배분 + cap 샘플링 함수
# =========================

def stratified_sampling_with_cap(
    df_success,
    df_fail,
    total_n=100,
    cap=10,
    random_state=42
):
    """
    성공/실패 원본 비율을 유지하면서 샘플링하되,
    각 성공/실패 데이터 내부에서 채널별 cap을 먼저 적용한다.
    """

    # 원본 성공/실패 비율 계산
    total = len(df_success) + len(df_fail)

    success_n = round(total_n * len(df_success) / total)
    fail_n    = total_n - success_n

    # 채널별 cap 적용
    df_success_capped = apply_channel_cap(df_success, cap=cap, random_state=random_state)
    df_fail_capped    = apply_channel_cap(df_fail, cap=cap, random_state=random_state)

    # cap 적용 후 실제 가능한 수로 조정
    success_n = min(success_n, len(df_success_capped))
    fail_n    = min(fail_n, len(df_fail_capped))

    # 샘플링
    sample_success = df_success_capped.sample(
        n=success_n,
        random_state=random_state
    )

    sample_fail = df_fail_capped.sample(
        n=fail_n,
        random_state=random_state
    )

    return sample_success, sample_fail


# =========================
# 7. 샘플링 실행
# =========================

TOTAL_N_PER_DOMAIN = 100
CHANNEL_CAP = 10

fnb_sample_success, fnb_sample_fail = stratified_sampling_with_cap(
    df_fnb_success,
    df_fnb_fail,
    total_n=TOTAL_N_PER_DOMAIN,
    cap=CHANNEL_CAP,
    random_state=RANDOM_STATE
)

it_sample_success, it_sample_fail = stratified_sampling_with_cap(
    df_it_success,
    df_it_fail,
    total_n=TOTAL_N_PER_DOMAIN,
    cap=CHANNEL_CAP,
    random_state=RANDOM_STATE
)


# =========================
# 8. 결과 합치기
# =========================

sample_all = pd.concat(
    [
        fnb_sample_success,
        fnb_sample_fail,
        it_sample_success,
        it_sample_fail
    ],
    ignore_index=True
)

# 중복 video_id 제거
before = len(sample_all)
sample_all = sample_all.drop_duplicates(subset=["video_id"]).reset_index(drop=True)
after = len(sample_all)
if before != after:
    print(f"[경고] 중복 video_id {before - after}개 제거 → 최종 {after}개")


# =========================
# 9. 결과 저장
# =========================

fnb_sample_success.to_csv(
    OUTPUT_DIR / "sample_fnb_shorts_success.csv",
    index=False,
    encoding="utf-8-sig"
)

fnb_sample_fail.to_csv(
    OUTPUT_DIR / "sample_fnb_shorts_fail.csv",
    index=False,
    encoding="utf-8-sig"
)

it_sample_success.to_csv(
    OUTPUT_DIR / "sample_it_shorts_success.csv",
    index=False,
    encoding="utf-8-sig"
)

it_sample_fail.to_csv(
    OUTPUT_DIR / "sample_it_shorts_fail.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_all.to_csv(
    OUTPUT_DIR / "sample_shorts_all_for_video_agent.csv",
    index=False,
    encoding="utf-8-sig"
)

print("샘플링 완료")
print(f"FnB 성공: {len(fnb_sample_success)}개")
print(f"FnB 실패: {len(fnb_sample_fail)}개")
print(f"IT 성공: {len(it_sample_success)}개")
print(f"IT 실패: {len(it_sample_fail)}개")
print(f"전체: {len(sample_all)}개")

print("\n저장 위치:")
print(OUTPUT_DIR / "sample_shorts_all_for_video_agent.csv")

샘플링 완료
FnB 성공: 48개
FnB 실패: 52개
IT 성공: 42개
IT 실패: 58개
전체: 200개

저장 위치:
data\sample_shorts_all_for_video_agent.csv


# 샘플링 결과 검증 코드


In [2]:
# 전체 요약
summary = sample_all.groupby(["domain", "success_label"]).agg(
    영상수=("video_id", "count"),
    채널수=("채널명", "nunique")
).reset_index()

display(summary)

# 채널별 포함 개수 확인
channel_dist = sample_all.groupby(["domain", "success_label", "채널명"]).size().reset_index(name="샘플수")
channel_dist = channel_dist.sort_values(["domain", "success_label", "샘플수"], ascending=[True, True, False])

display(channel_dist)

# 채널 cap 위반 확인
cap_violation = channel_dist[channel_dist["샘플수"] > CHANNEL_CAP]

print("cap 위반 채널 수:", len(cap_violation))
display(cap_violation)

,domain,success_label,영상수,채널수
0,FnB,fail,52,21
1,FnB,success,48,28
2,IT,fail,58,11
3,IT,success,42,17


,domain,success_label,채널명,샘플수
15,FnB,fail,오뚜기 Daily,6
0,FnB,fail,CJ제일제당,5
1,FnB,fail,CU [씨유튜브],4
2,FnB,fail,Coca-Cola Korea,4
8,FnB,fail,농심 nongshim,4
...,...,...,...,...
71,IT,success,만나플러스,2
64,IT,success,LG유플러스 (LG Uplus),1
65,IT,success,NOL,1
66,IT,success,SK브로드밴드_B tv,1


cap 위반 채널 수: 0


,domain,success_label,채널명,샘플수


업로드 기간도 확인하려면:

In [3]:
sample_all["업로드일시"] = pd.to_datetime(sample_all["업로드일시"], errors="coerce")
sample_all["upload_year"] = sample_all["업로드일시"].dt.year

year_summary = sample_all.groupby(["domain", "success_label", "upload_year"]).size().reset_index(name="영상수")
display(year_summary)

,domain,success_label,upload_year,영상수
0,FnB,fail,2021,2
1,FnB,fail,2022,5
2,FnB,fail,2023,18
3,FnB,fail,2024,13
4,FnB,fail,2025,14
5,FnB,success,2022,4
6,FnB,success,2023,8
7,FnB,success,2024,7
8,FnB,success,2025,16
9,FnB,success,2026,13


채널 규모가 있다면:

In [4]:
if "구독자수" in sample_all.columns:
    def classify_channel_size(sub):
        if sub < 1000:
            return "0_기타"
        elif sub < 10000:
            return "1_나노"
        elif sub < 50000:
            return "2_마이크로"
        elif sub < 100000:
            return "3_미드"
        elif sub < 500000:
            return "4_매크로"
        else:
            return "5_메가"

    sample_all["채널규모티어"] = sample_all["구독자수"].apply(classify_channel_size)

    tier_summary = sample_all.groupby(["domain", "success_label", "채널규모티어"]).size().reset_index(name="영상수")
    display(tier_summary)

,domain,success_label,채널규모티어,영상수
0,FnB,fail,1_나노,1
1,FnB,fail,2_마이크로,8
2,FnB,fail,3_미드,10
3,FnB,fail,4_매크로,27
4,FnB,fail,5_메가,6
5,FnB,success,0_기타,7
6,FnB,success,1_나노,15
7,FnB,success,2_마이크로,9
8,FnB,success,3_미드,5
9,FnB,success,4_매크로,7
